# Cluster Analysis of Selected Language Sample

### Common Setup

In [ ]:
# Setting system path and project root
import os
import sys

PROJECT_ROOT_DIR = os.path.abspath('../../')
sys.path.append(PROJECT_ROOT_DIR) # bringing system path to project root

def get_fp(relative_path):
    return os.path.join(PROJECT_ROOT_DIR, relative_path)

### Reading Language Sample with Resource Category Data

In [ ]:
import pandas as pd

csv_path = get_fp("data_dir/fineweb2_metadata/l2v_nested_samples/200.csv")
sample_lang_doc_distr_df = pd.read_csv(csv_path)

print(f'Shape of sample dataframe: {sample_lang_doc_distr_df.shape}')

columns_of_interest = ["subset", "code", "name", "script", "family"]

for col in columns_of_interest:
    if col in sample_lang_doc_distr_df.columns:
        uniques = sample_lang_doc_distr_df[col].dropna().unique()   # drop NaN, then get uniques
        print(f"\nUnique values in '{col}' ({len(uniques)} total):")
        # print(uniques)
    else:
        print(f"\nColumn '{col}' not found in the CSV.")

### Extracting Distances Matrices

In [ ]:
from lang2vec.lang2vec import *
def get_dist_matrix(lang_codes, distance_name):
        with zf(DISTANCES_FILE, 'r') as zp:
                data = sparse.load_npz(zp.open(map_distance_to_filename(distance_name)))

        present_idxs = np.array([DISTANCE_LANGUAGES.index(code) for code in lang_codes])
        dist_mat = [[data[lang2_idx, lang1_idx] if lang1_idx > lang2_idx else data[lang1_idx, lang2_idx]  for lang2_idx in present_idxs] for lang1_idx in present_idxs]
        return dist_mat

In [ ]:
# Fetch distance matrix
eval_distances = {
    "genetic",
    "geographic", 
    "syntactic",
    "inventory",
    "phonological",
    "featural"
}

# Lang2Vec provides a collection of supported language identifiers
available_langs = set(DISTANCE_LANGUAGES)          # e.g. {'eng', 'fra', ...}

# Ensure codes are strings for reliable comparison
lang_codes = sample_lang_doc_distr_df['code'].astype(str).unique()

dist_matrix_map = {distance_name: get_dist_matrix(lang_codes, distance_name) for distance_name in eval_distances}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## Combine the six distance matrices into one
# `dist_matrix_map` is a dict: {distance_name: matrix}
# All matrices have the same shape (n_langs × n_langs) and are symmetric.

def combine_distance_matrices(dist_map, weights=None):
    """Return a single distance matrix that is a weighted average of the inputs.

    The function accepts dictionaries whose values may be Python lists or
    NumPy arrays; they are all cast to ``np.ndarray`` internally.
    """
    # Convert every distance matrix to a NumPy array (ensures ``.shape`` exists)
    arrays = {name: np.asarray(mat, dtype=float) for name, mat in dist_map.items()}

    distance_names = list(arrays.keys())
    n = arrays[distance_names[0]].shape[0]               # number of languages

    # Default: equal weight for each distance type
    if weights is None:
        weights = np.ones(len(distance_names)) / len(distance_names)
    else:
        weights = np.asarray(weights, dtype=float)
        weights = weights / weights.sum()                # normalise

    combined = np.zeros((n, n), dtype=float)
    for w, name in zip(weights, distance_names):
        combined += w * arrays[name]

    # Enforce exact symmetry & zero diagonal (important for MDS)
    combined = (combined + combined.T) / 2.0
    np.fill_diagonal(combined, 0.0)

    return combined

combined_dist = combine_distance_matrices(dist_matrix_map)


## Project to Euclidean space with Classical MDS
# MDS with `dissimilarity='precomputed'` gives us coordinates that
# preserve the combined distances as well as possible.
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, metric=True)
coords = mds.fit_transform(combined_dist)   # shape: (n_langs, 2)

## Evaluate K‑means for a range of k
range_k = list(range(2, len(lang_codes)))
silhouette_scores = []
inertias = []

for k in range_k:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(coords)

    sil = silhouette_score(coords, labels, metric='euclidean')
    silhouette_scores.append(sil)

    inertias.append(km.inertia_)   # sum of squared distances to centroids

## Plot Silhouette & Elbow curves
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Silhouette curve
ax[0].plot(range_k, silhouette_scores, marker='o')
ax[0].set_xlabel('Number of clusters (k)')
ax[0].set_ylabel('Silhouette score')
ax[0].set_title('Silhouette analysis')
ax[0].grid(True)

# Elbow (inertia) curve
ax[1].plot(range_k, inertias, marker='o')
ax[1].set_xlabel('Number of clusters (k)')
ax[1].set_ylabel('Inertia')
ax[1].set_title('Elbow method')
ax[1].grid(True)

plt.tight_layout()
plt.show()

## Print the highest silhouette score and the corresponding k
best_idx = np.argmax(silhouette_scores)          # position of the max
best_k   = range_k[best_idx]                     # number of clusters that gave it
best_score = silhouette_scores[best_idx]

print(f"Highest silhouette score: {best_score:.4f} (achieved with k = {best_k})")

In [ ]:
## Obsolete testing code
# DISTANCE_LANGUAGES
# data[0, 4]
# #dist_mat
# genetic_distance('rus','ajz')
# present_codes
# random_index = 44
# lang_sel = present_codes[random_index]
# print(lang_sel)
# lang_ind = present_idxs[random_index]
# print(lang_ind)
# print(dist_mat[random_index])
# ext_gen_dist = [genetic_distance(lang, present_codes[random_index]) for lang in present_codes]
# print(ext_gen_dist)
# print(ext_gen_dist == dist_mat[random_index])
# dist_mat = np.asarray(dist_mat, dtype=np.float64)
# print(dist_mat)